# Week 7 — Merging DataFrames: Full Pipeline & Payment Analysis
## Thursday Exercises | Phase 2a Python | PORA Academy Cohort 7

Today you chain several merges into one complete analysis table, bring in
reviews and payments, and answer real business questions about the Olist store.

**Instructions:**
- Run the **Data Setup** cell first — every question depends on it.
- Read each question carefully and check your output against the **Expected** value shown.
- For DeepSeek-marked questions, write the prompt yourself, then *verify* the result — never paste blindly.
- Save your notebook before submitting.

## Data Setup

In [1]:
import pandas as pd
from google.colab import drive
import os

#drive.mount('/content/drive')
olist_path = '/content/drive/MyDrive/olist-data'

orders       = pd.read_csv(os.path.join(olist_path, '/content/olist_orders_dataset.csv'))
customers    = pd.read_csv(os.path.join(olist_path, '/content/olist_customers_dataset.csv'))
order_items  = pd.read_csv(os.path.join(olist_path, '/content/olist_order_items_dataset.csv'))
products     = pd.read_csv(os.path.join(olist_path, '/content/olist_products_dataset.csv'))
reviews      = pd.read_csv(os.path.join(olist_path, '/content/olist_order_reviews_dataset.csv'))
payments     = pd.read_csv(os.path.join(olist_path, '/content/olist_order_payments_dataset.csv'))
translations = pd.read_csv(os.path.join(olist_path, '/content/product_category_name_translation.csv'))

print(f"orders {orders.shape} | customers {customers.shape} | items {order_items.shape}")
print(f"products {products.shape} | reviews {reviews.shape} | payments {payments.shape}")

orders (99441, 8) | customers (99441, 5) | items (112650, 7)
products (32951, 9) | reviews (99224, 7) | payments (103886, 5)


## Using AI Assistance (DeepSeek)

Some questions below are marked **(DeepSeek)**. When you use DeepSeek this week, follow the protocol:

1. **Understand first** — describe in plain English what you want *before* you prompt.
2. **Prompt with context** — name the DataFrame, list the relevant columns, and state the exact question.
3. **Validate the output** — compare every result against the expected value in the question. If it doesn't match, the *code* is wrong, not the curriculum.
4. **Never copy blindly** — you must be able to explain every line DeepSeek gives you.

## Question 1 — Build the Full Merged Table

Chain the merges into one analysis table, in this order:

`orders` → merge `customers` (on `customer_id`) → merge `order_items` (on `order_id`)
→ merge `products[['product_id', 'product_category_name']]` (on `product_id`, `how='left'`)
→ merge `translations` (on `product_category_name`, `how='left'`)

Call the result `full` and print its shape.

**Expected shape:** `(112650, 20)`

In [2]:
# Q1 — chain orders + customers + order_items + products + translations
full = (
    orders
    .merge(customers, on='customer_id')
    .merge(order_items, on='order_id')
    .merge(products[['product_id', 'product_category_name']], on='product_id', how='left')
    .merge(translations, on='product_category_name', how='left')
)

print(full.shape)  # expected: (112650, 20)

(112650, 20)


## Question 2 — Total Revenue by State

Using the `full` table from Question 1, group by `customer_state` and sum the
`price` column to get total revenue per state. Sort from highest to lowest and
print the top 5.

**Expected:** SP is #1. Total revenue from **SP** = **R$5,202,955.05**

In [6]:
top5_states_revenue = (
    full.groupby('customer_state')['price']
        .sum()
        .sort_values(ascending=False)
        .head(5)
)


print(top5_states_revenue.head(5).round(2))
print(f"SP total revenue: R${top5_states_revenue['SP']:,.2f}")  # expected: R$5,202,955.05

customer_state
SP    5202955.05
RJ    1824092.67
MG    1585308.03
RS     750304.02
PR     683083.76
Name: price, dtype: float64
SP total revenue: R$5,202,955.05


## Question 3 — Category Revenue with Reviews

Merge `reviews[['order_id', 'review_score']]` into `full` with a **left** join on
`order_id` (call it `full_reviews`). Then group by `product_category_name_english`
and compute, for each category:

- `total_revenue` = sum of `price`
- `avg_review` = mean of `review_score`
- `order_count` = number of unique `order_id`

Sort by `total_revenue` descending and print the top 5.

**Expected top 5 by revenue:**

| category | total_revenue |
|---|---|
| health_beauty | 1,258,681.34 |
| watches_gifts | 1,205,005.68 |
| bed_bath_table | 1,036,988.68 |
| sports_leisure | 988,048.97 |
| computers_accessories | 911,954.32 |

In [8]:
# Q3 — merge reviews (left, on order_id), then category-level revenue + avg review + order count
full_reviews = full.merge(reviews[['order_id', 'review_score']], on='order_id', how='left')

cat_analysis = (
    full_reviews.groupby('product_category_name_english')
        .agg(
            total_revenue=('price', 'sum'),
            avg_review=('review_score', 'mean'),
            order_count=('order_id', 'nunique')
        )
        .sort_values('total_revenue', ascending=False)
)

print(cat_analysis.head(5).round(2))  # expected top: health_beauty 1258681.34

                               total_revenue  avg_review  order_count
product_category_name_english                                        
health_beauty                     1263138.54        4.14         8836
watches_gifts                     1206075.33        4.02         5624
bed_bath_table                    1050936.61        3.90         9417
sports_leisure                     993656.51        4.11         7720
computers_accessories              919640.54        3.93         6689


## Question 4 — Payment Type Breakdown

Using the `payments` table, group by `payment_type` and compute:

- `count` = number of payment rows
- `avg_installments` = mean of `payment_installments`

Then add a `pct` column = each type's share of the total count (× 100).
Sort by `count` descending and print the result.

**Expected:**

| payment_type | count | pct | avg_installments |
|---|---|---|---|
| credit_card | 76,795 | 73.9% | 3.0 |
| boleto | 19,784 | 19.0% | — |
| voucher | 5,775 | 5.6% | — |
| debit_card | 1,529 | 1.5% | — |

In [10]:
# Q4 — payment_type breakdown: count, avg_installments, pct share
pay_summary = (
    payments.groupby('payment_type')
        .agg(
            count=('payment_type', 'size'),
            avg_installments=('payment_installments', 'mean')
        )
        .sort_values('count', ascending=False)
)

pay_summary['pct'] = (pay_summary['count'] / pay_summary['count'].sum() * 100).round(1)


print(pay_summary.round(2))  # expected: credit_card 76795 (73.9%), avg 3.0 installments

              count  avg_installments   pct
payment_type                               
credit_card   76795              3.51  73.9
boleto        19784              1.00  19.0
voucher        5775              1.00   5.6
debit_card     1529              1.00   1.5
not_defined       3              1.00   0.0


## Question 5 — Group Exercise (40 min)

Work with your group to build a complete multi-table analysis.

**Task 1 — State analysis table.**
Starting from `full_reviews` (from Question 3), group by `customer_state` and compute:
- `total_revenue` = sum of `price`
- `avg_review_score` = mean of `review_score`
- `order_count` = number of unique `order_id`

**Task 2 — Best-reviewed large state.**
Among states with **more than 500 orders**, which state has the **highest**
`avg_review_score`? Filter to `order_count > 500`, then sort by `avg_review_score`.

**Task 3 — (DeepSeek) Average installments by state.**
Ask DeepSeek to help you merge the `payments` table into the analysis and show the
**average `payment_installments` by state**. Provide the DataFrame names and join keys
in your prompt.
**Verify:** the overall (all-Brazil) average `payment_installments` = **2.85**.

In [11]:
# ── Task 1: State analysis table ────────────────────────────────
state_summary = (
    full_reviews.groupby('customer_state')
        .agg(
            total_revenue=('price', 'sum'),
            avg_review_score=('review_score', 'mean'),
            order_count=('order_id', 'nunique')
        )
        .sort_values('total_revenue', ascending=False)
)

print(state_summary)

                total_revenue  avg_review_score  order_count
customer_state                                              
SP                 5228869.17          4.126859        41375
RJ                 1831678.85          3.807092        12762
MG                 1591518.47          4.086957        11544
RS                  754250.33          4.054270         5432
PR                  685911.51          4.104603         4998
SC                  522120.11          4.003128         3612
BA                  513182.78          3.814392         3358
DF                  304658.17          4.005381         2125
GO                  297535.49          3.993092         2007
ES                  275910.68          3.992848         2025
PE                  263801.64          3.957127         1648
CE                  227931.60          3.805707         1327
PA                  179429.86          3.791940          970
MT                  156741.23          3.978032          903
MA                  1198

In [12]:
# ── Task 2: Best-reviewed large state ───────────────────────────
large_states = state_summary[state_summary['order_count'] > 500]
best_reviewed = large_states.sort_values('avg_review_score', ascending=False)

print(best_reviewed.head())
print("\nTop state:", best_reviewed.index[0],
      "-", round(best_reviewed['avg_review_score'].iloc[0], 3))

                total_revenue  avg_review_score  order_count
customer_state                                              
SP                 5228869.17          4.126859        41375
PR                  685911.51          4.104603         4998
MG                 1591518.47          4.086957        11544
RS                  754250.33          4.054270         5432
MS                  118217.71          4.053076          709

Top state: SP - 4.127


In [13]:
# Build one row per order first, to avoid double-counting from order_items
orders_states = full[['order_id', 'customer_state']].drop_duplicates()

orders_payments = orders_states.merge(payments, on='order_id', how='left')

# Overall check
overall_avg = orders_payments['payment_installments'].mean()
print("Overall avg installments:", round(overall_avg, 2))  # should be ~2.85

# By state
installments_by_state = (
    orders_payments.groupby('customer_state')['payment_installments']
        .mean()
        .sort_values(ascending=False)
)
print(installments_by_state)

Overall avg installments: 2.86
customer_state
PB    3.761484
AL    3.712941
SE    3.639437
RN    3.568401
AC    3.559524
PE    3.486941
CE    3.483081
RO    3.466667
PI    3.444444
AM    3.287582
PA    3.215920
BA    3.192919
MA    3.067105
MT    3.042977
ES    2.994757
MG    2.984669
RS    2.977454
TO    2.973333
RJ    2.965684
GO    2.953788
MS    2.900000
SC    2.868830
PR    2.860956
RR    2.739130
DF    2.728062
SP    2.621992
AP    2.614286
Name: payment_installments, dtype: float64


In [14]:
# Task 1 — groupby customer_state: total_revenue, avg_review_score, order_count
state_summary = (
    full_reviews.groupby('customer_state')
        .agg(
            total_revenue=('price', 'sum'),
            avg_review_score=('review_score', 'mean'),
            order_count=('order_id', 'nunique')
        )
        .sort_values('total_revenue', ascending=False)
)

print(state_summary)
print()

# Task 2 — among states with order_count > 500, which has the highest avg_review_score?
large_states = state_summary[state_summary['order_count'] > 500]
best_reviewed = large_states.sort_values('avg_review_score', ascending=False)

print(best_reviewed.head())
print("\nTop state:", best_reviewed.index[0],
      "-", round(best_reviewed['avg_review_score'].iloc[0], 3))
print()

# Task 3 (DeepSeek) — merge payments, average payment_installments by state
# Write your DeepSeek prompt, then verify the all-Brazil average = 2.85
# Build one row per order first, to avoid double-counting from order_items
orders_states = full[['order_id', 'customer_state']].drop_duplicates()

orders_payments = orders_states.merge(payments, on='order_id', how='left')

# Overall check
overall_avg = orders_payments['payment_installments'].mean()
print("Overall avg installments:", round(overall_avg, 2))  # should be ~2.85

# By state
installments_by_state = (
    orders_payments.groupby('customer_state')['payment_installments']
        .mean()
        .sort_values(ascending=False)
)
print(installments_by_state)

                total_revenue  avg_review_score  order_count
customer_state                                              
SP                 5228869.17          4.126859        41375
RJ                 1831678.85          3.807092        12762
MG                 1591518.47          4.086957        11544
RS                  754250.33          4.054270         5432
PR                  685911.51          4.104603         4998
SC                  522120.11          4.003128         3612
BA                  513182.78          3.814392         3358
DF                  304658.17          4.005381         2125
GO                  297535.49          3.993092         2007
ES                  275910.68          3.992848         2025
PE                  263801.64          3.957127         1648
CE                  227931.60          3.805707         1327
PA                  179429.86          3.791940          970
MT                  156741.23          3.978032          903
MA                  1198

## Weekly Assignment 7

Submit `week7_assignment.ipynb`:

1. Build the full merged table (orders + customers + items + products + translation).
   Verify final shape = **(112650, 20)**

2. From the full merged table, answer:
   - Which state has the highest average price per item? *(Groupby state, mean of price)*
   - What is the total revenue from "health_beauty" category? *(Expected: R$1,258,681.34)*

3. Merge in reviews. Find the 5 **worst-rated** product categories (min 100 orders).
   *(Expected bottom 5 include security_and_services at 2.50 avg)*

4. **DeepSeek challenge:** Ask DeepSeek to merge orders + payments and calculate total
   payment value per order status. What is the total payment value for 'delivered'
   orders? Paste your prompt and verify the output is internally consistent.